# TAU2 Task Sampling with Vector Similarity

This notebook demonstrates how to implement a Python function to:

1. **Filter tasks**: For every task in `tasks.json`, filter them out from `tasks_full.json` to create a `task_excluded` list
2. **Vector similarity search**: For each task in `tasks.json`, use vector embeddings to find the most similar task in the `task_excluded` list
3. **Export results**: Save the sampled tasks to a new JSON file

We'll use sentence-transformers for creating embeddings and cosine similarity for finding the most similar tasks based on the `user_scenario` content.

## 1. Import Required Libraries

Import Python libraries for data processing, vector operations, and embedding models.

In [1]:
import json
import os
import numpy as np
import pandas as pd
from typing import List, Dict, Any, Tuple
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("All libraries imported successfully!")

/home/lijiah/workspace/tau2-bench/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries imported successfully!


## 2. Load tasks.json and tasks_full.json

Load the task files and examine their structure.

In [2]:
# Define file paths
tasks_json_path = "/home/lijiah/workspace/tau2-bench/data/tau2/domains/telecom/tasks.json"
tasks_full_json_path = "/home/lijiah/workspace/tau2-bench/data/tau2/domains/telecom/tasks_full.json"
output_path = "/home/lijiah/workspace/tau2-bench/data/tau2/domains/telecom/tasks_sampled.json"

# Load tasks.json
with open(tasks_json_path, 'r', encoding='utf-8') as f:
    tasks_selected = json.load(f)

# Load tasks_full.json
with open(tasks_full_json_path, 'r', encoding='utf-8') as f:
    tasks_full = json.load(f)

print(f"Loaded {len(tasks_selected)} selected tasks from tasks.json")
print(f"Loaded {len(tasks_full)} full tasks from tasks_full.json")
print(f"Output will be saved to: {output_path}")

# Display sample structure
print("\nSample task structure from tasks.json:")
print(f"Task ID: {tasks_selected[0]['id']}")
print(f"Has user_scenario: {'user_scenario' in tasks_selected[0]}")
print(f"Has ticket: {'ticket' in tasks_selected[0]}")

Loaded 114 selected tasks from tasks.json
Loaded 2285 full tasks from tasks_full.json
Output will be saved to: /home/lijiah/workspace/tau2-bench/data/tau2/domains/telecom/tasks_sampled.json

Sample task structure from tasks.json:
Task ID: [mobile_data_issue]data_mode_off|data_usage_exceeded[PERSONA:None]
Has user_scenario: True
Has ticket: True


## 3. Extract Task IDs and Build task_excluded

Filter out tasks from `tasks_full.json` that exist in `tasks.json` to create the excluded list.

In [3]:
# Extract task IDs from tasks.json
selected_task_ids = {task['id'] for task in tasks_selected}
print(f"Selected task IDs count: {len(selected_task_ids)}")

# Filter out selected tasks from tasks_full to create task_excluded
task_excluded = [task for task in tasks_full if task['id'] not in selected_task_ids]

print(f"Tasks excluded count: {len(task_excluded)}")
print(f"Total tasks (selected + excluded): {len(tasks_selected) + len(task_excluded)}")
print(f"Original tasks_full count: {len(tasks_full)}")

# Verify filtering worked correctly
assert len(tasks_selected) + len(task_excluded) == len(tasks_full), "Filtering error: counts don't match"
print("✓ Filtering completed successfully!")

Selected task IDs count: 114
Tasks excluded count: 2171
Total tasks (selected + excluded): 2285
Original tasks_full count: 2285
✓ Filtering completed successfully!


## 4. Prepare Embeddings for Vector Search

Initialize the sentence transformer model for creating embeddings.

In [4]:
# Install required packages if needed and configure SSL
import subprocess
import sys

def install_package(package):
    """Install a package if it's not already installed"""
    try:
        __import__(package)
        print(f"✓ {package} is already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✓ {package} installed successfully")

# Try to install sentence-transformers (may fail due to SSL)
try:
    install_package("sentence_transformers")
except Exception as e:
    print(f"Note: sentence-transformers installation failed: {e}")
    print("Will use TF-IDF as fallback (already available with scikit-learn)")

# Configure SSL settings for potential issues
import ssl
import os

# Disable SSL verification for development (use with caution in production)
ssl._create_default_https_context = ssl._create_unverified_context

print("✓ SSL configuration updated")

✓ sentence_transformers is already installed
✓ SSL configuration updated


In [5]:
# Handle SSL certificate issues with multiple fallback options
import ssl
import urllib.request
from sklearn.feature_extraction.text import TfidfVectorizer

def try_load_sentence_transformer():
    """Try to load SentenceTransformer with SSL workarounds"""
    # try:
    # Method 1: Try with SSL context disabled (for local/development use)
    ssl_context = ssl.create_default_context()
    ssl_context.check_hostname = False
    ssl_context.verify_mode = ssl.CERT_NONE
    
    # Temporarily disable SSL verification
    import os
    os.environ['CURL_CA_BUNDLE'] = ''
    os.environ['REQUESTS_CA_BUNDLE'] = ''
    model_name = 'all-MiniLM-L6-v2'
    print(f"Trying to load sentence transformer model: {model_name}")
    
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(model_name, trust_remote_code=True)
    print("✓ SentenceTransformer model loaded successfully!")
    return model, "sentence_transformer"
        
    # except Exception as e:
    #     print(f"SentenceTransformer failed: {e}")
    #     print("Falling back to TF-IDF vectorizer...")
        
    #     # Fallback: Use TF-IDF vectorizer (no internet required)
    #     vectorizer = TfidfVectorizer(
    #         max_features=1000,
    #         stop_words='english',
    #         ngram_range=(1, 2),
    #         lowercase=True
    #     )
    #     print("✓ TF-IDF vectorizer initialized successfully!")
    #     return vectorizer, "tfidf"

# Load the model/vectorizer
model, model_type = try_load_sentence_transformer()
print(f"Using {model_type} for embeddings")

# Function to extract user_scenario text for embedding
def extract_user_scenario_text(task: Dict[str, Any]) -> str:
    """Extract text from user_scenario for embedding."""
    user_scenario = task.get('user_scenario', {})
    
    # Extract all relevant text from user_scenario
    text_parts = []
    
    # Add persona if exists
    if user_scenario.get('persona'):
        text_parts.append(f"Persona: {user_scenario['persona']}")
    
    # Add instructions
    instructions = user_scenario.get('instructions', {})
    if instructions:
        for key, value in instructions.items():
            if value and isinstance(value, str):
                text_parts.append(f"{key}: {value}")
    
    # Also include ticket information as it's part of the scenario context
    if task.get('ticket'):
        text_parts.append(f"Ticket: {task['ticket']}")
    
    return " ".join(text_parts)

# Test the function
sample_text = extract_user_scenario_text(tasks_selected[0])
print(f"\nSample extracted text (first 200 chars):")
print(sample_text[:200] + "..." if len(sample_text) > 200 else sample_text)

Trying to load sentence transformer model: all-MiniLM-L6-v2


/home/lijiah/workspace/tau2-bench/.venv/lib/python3.10/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/home/lijiah/workspace/tau2-bench/.venv/lib/python3.10/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/home/lijiah/workspace/tau2-bench/.venv/lib/python3.10/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings


✓ SentenceTransformer model loaded successfully!
Using sentence_transformer for embeddings

Sample extracted text (first 200 chars):
domain: telecom reason_for_call: You mobile data is not working properly. It either stops working or is very slow. You want to fix it and absolutely want to get excellent internet speed on your phone....


## 5. Compute Embeddings for user_scenario

Create embeddings for the user_scenario content of tasks in both lists.

In [6]:
# Extract text for selected tasks
print("Extracting text from selected tasks...")
selected_texts = [extract_user_scenario_text(task) for task in tasks_selected]

# Extract text for excluded tasks  
print("Extracting text from excluded tasks...")
excluded_texts = [extract_user_scenario_text(task) for task in task_excluded]

# Create embeddings based on the model type
if model_type == "sentence_transformer":
    print("Creating embeddings using SentenceTransformer...")
    # Create embeddings for selected tasks
    selected_embeddings = model.encode(selected_texts, show_progress_bar=True)
    # Create embeddings for excluded tasks
    excluded_embeddings = model.encode(excluded_texts, show_progress_bar=True)
    
elif model_type == "tfidf":
    print("Creating embeddings using TF-IDF...")
    # Combine all texts to fit the vectorizer
    all_texts = selected_texts + excluded_texts
    
    # Fit and transform all texts
    all_embeddings = model.fit_transform(all_texts).toarray()
    
    # Split back into selected and excluded
    selected_embeddings = all_embeddings[:len(selected_texts)]
    excluded_embeddings = all_embeddings[len(selected_texts):]

print(f"✓ Selected embeddings shape: {selected_embeddings.shape}")
print(f"✓ Excluded embeddings shape: {excluded_embeddings.shape}")
print(f"✓ Using {model_type} for similarity computation")

Extracting text from selected tasks...
Extracting text from excluded tasks...
Creating embeddings using SentenceTransformer...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]/home/lijiah/workspace/tau2-bench/.venv/lib/python3.10/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/home/lijiah/workspace/tau2-bench/.venv/lib/python3.10/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Batches: 100%|██████████| 4/4 [00:00<00:00, 13.28it/s]

Batches: 100%|██████████| 68/68 [00:01<00:00, 44.53it/s]

✓ Selected embeddings shape: (114, 384)
✓ Excluded embeddings shape: (2171, 384)
✓ Using sentence_transformer for similarity computation


## 6. Find Top-1 Most Similar Tasks for Each Task

Use cosine similarity to find the most similar excluded task for each selected task.

In [16]:
# Calculate similarity matrix between selected and excluded tasks
print("Calculating cosine similarity matrix...")
similarity_matrix = cosine_similarity(selected_embeddings, excluded_embeddings)
print(f"✓ Similarity matrix shape: {similarity_matrix.shape}")

# Find most similar tasks
task_sampled = []
used_indices = set()

print("\nFinding most similar tasks...")
for i, selected_task in enumerate(tasks_selected):
    # Get similarity scores for this selected task
    similarities = similarity_matrix[i]

    # Sort excluded tasks by similarity (descending)
    sorted_indices = np.argsort(similarities)[::-1]
    print(sorted_indices)

    # Find the most similar task that hasn't been used yet
    for idx in sorted_indices:
        # if similarities[idx]>0.999:  # Skip exact matches (if any)
        #     continue  # Skip exact matches (if any)
        # Check if this index has already been used
        if idx not in used_indices:
            task_sampled.append(task_excluded[idx])
            used_indices.add(idx)
            
            # Print similarity info for first few examples
            if i < 5:  # Show details for first 5 tasks
                similarity_score = similarities[idx]
                print(f"  Selected: {selected_task['id']}")
                print(f"  → Similar: {task_excluded[idx]['id']}")
                print(f"  → Similarity: {similarity_score:.4f}")
                print()
            break
    else:
        print(f"Warning: No unused similar task found for {selected_task['id']}")

print(f"✓ Sampled {len(task_sampled)} similar tasks")
print(f"✓ Used {len(used_indices)} unique excluded tasks")

Calculating cosine similarity matrix...
✓ Similarity matrix shape: (114, 2171)

Finding most similar tasks...
[  6 122 124 ... 849 780 712]
  Selected: [mobile_data_issue]data_mode_off|data_usage_exceeded[PERSONA:None]
  → Similar: [mobile_data_issue]bad_vpn[PERSONA:None]
  → Similarity: 1.0000

[  6 122 124 ... 849 780 712]
  Selected: [mobile_data_issue]airplane_mode_on|data_mode_off[PERSONA:None]
  → Similar: [mobile_data_issue]airplane_mode_on|bad_vpn|data_mode_off|data_usage_exceeded[PERSONA:None]
  → Similarity: 1.0000

[ 101  135   34 ...  364 2075  367]
  Selected: [mobile_data_issue]airplane_mode_on|user_abroad_roaming_enabled_off[PERSONA:None]
  → Similar: [mobile_data_issue]airplane_mode_on|bad_vpn|data_mode_off|user_abroad_roaming_disabled_on[PERSONA:None]
  → Similarity: 1.0000

[  4 119  81 ... 218 235 233]
  Selected: [mobile_data_issue]data_saver_mode_on|data_usage_exceeded[PERSONA:Easy]
  → Similar: [mobile_data_issue]data_saver_mode_on[PERSONA:Easy]
  → Similarity: 1.

In [ ]:
    "id": "[mobile_data_issue]bad_vpn[PERSONA:None]",
    "description": {
      "purpose": "Test resolution path: Mobile Data/Slow Internet Issues.",
      "relevant_policies": null,
      "notes": null
    },
    "user_scenario": {
      "persona": null,
      "instructions": {
        "domain": "telecom",
        "reason_for_call": "You mobile data is not working properly. It either stops working or is very slow. You want to fix it and absolutely want to get excellent internet speed on your phone. You are not willing to accept any other internet speed (poor, fair or good). You do not have access to wifi.",
        "known_info": "You are John Smith with phone number 555-123-2002. You are currently at home in the United States.",
        "unknown_info": null,
        "task_instructions": "If the agent suggests actions that don't immediately fix the issue, follow their guidance but express mild frustration after the first unsuccessful attempt. You will consider the issue resolved only when speed test returns excellent internet speed and nothing else. If it returns poor, fair or good, you will not consider the issue resolved. You are willing to refuel 2.0 GB of data if necessary, but you do not want to change your mobile data plan. If the tool call does not return updated status information, you might need to perform another tool call to get the updated status. \nWhenever the agent asks you about your device, always ground your responses on the results of tool calls. \nFor example: If the agent asks what the status bar shows, always ground your response on the results of the `get_status_bar` tool call. If the agent asks if you are able to send an MMS message, always ground your response on the results of the `can_send_mms` tool call.\nNever make up the results of tool calls, always ground your responses on the results of tool calls.\nIf you are unsure about whether an action is necessary, always ask the agent for clarification.\n"

    "id": "[mobile_data_issue]data_mode_off|data_usage_exceeded[PERSONA:None]",
    "description": {
      "purpose": "Test resolution path: Mobile Data/Slow Internet Issues.",
      "relevant_policies": null,
      "notes": null
    },
    "user_scenario": {
      "persona": null,
      "instructions": {
        "domain": "telecom",
        "reason_for_call": "You mobile data is not working properly. It either stops working or is very slow. You want to fix it and absolutely want to get excellent internet speed on your phone. You are not willing to accept any other internet speed (poor, fair or good). You do not have access to wifi.",
        "known_info": "You are John Smith with phone number 555-123-2002. You are currently at home in the United States.",
        "unknown_info": null,
        "task_instructions": "If the agent suggests actions that don't immediately fix the issue, follow their guidance but express mild frustration after the first unsuccessful attempt. You will consider the issue resolved only when speed test returns excellent internet speed and nothing else. If it returns poor, fair or good, you will not consider the issue resolved. You are willing to refuel 2.0 GB of data if necessary, but you do not want to change your mobile data plan. If the tool call does not return updated status information, you might need to perform another tool call to get the updated status. \nWhenever the agent asks you about your device, always ground your responses on the results of tool calls. \nFor example: If the agent asks what the status bar shows, always ground your response on the results of the `get_status_bar` tool call. If the agent asks if you are able to send an MMS message, always ground your response on the results of the `can_send_mms` tool call.\nNever make up the results of tool calls, always ground your responses on the results of tool calls.\nIf you are unsure about whether an action is necessary, always ask the agent for clarification.\n"
      }



## 7. Export task_sampled to JSON File

Save the final list of sampled tasks to a new JSON file.

In [ ]:
# Save the sampled tasks to JSON file
print(f"Saving {len(task_sampled)} sampled tasks to {output_path}")

# Create output directory if it doesn't exist
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Save to JSON file with proper formatting
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(task_sampled, f, indent=2, ensure_ascii=False)

print("✓ Results saved successfully!")

# Verify the saved file
with open(output_path, 'r', encoding='utf-8') as f:
    saved_tasks = json.load(f)
    
print(f"✓ Verification: Loaded {len(saved_tasks)} tasks from saved file")

# Show sample of what was saved
print(f"\nSample saved task IDs:")
for i, task in enumerate(saved_tasks[:5]):
    print(f"  {i+1}. {task['id']}")
if len(saved_tasks) > 5:
    print(f"  ... and {len(saved_tasks) - 5} more tasks")

## Summary

This notebook successfully implemented the task sampling functionality:

1. ✅ **Loaded task files**: Loaded tasks.json and tasks_full.json
2. ✅ **Filtered excluded tasks**: Created task_excluded list by removing selected tasks from full list
3. ✅ **Generated embeddings**: Used sentence-transformers to create vector embeddings for user_scenario content
4. ✅ **Found similar tasks**: Used cosine similarity to find the most similar excluded task for each selected task
5. ✅ **Exported results**: Saved the sampled tasks to a new JSON file

### Key Features:
- **Vector similarity search**: Uses state-of-the-art sentence transformers for semantic similarity
- **Duplicate prevention**: Ensures no excluded task is sampled multiple times
- **Comprehensive text extraction**: Combines user_scenario, instructions, and ticket information
- **Progress tracking**: Shows detailed information about the sampling process

### Output:
- **File**: `tasks_sampled.json` containing the most similar tasks
- **Size**: Same number of tasks as in the original tasks.json
- **Quality**: Each sampled task is the most similar available excluded task

The sampled tasks can now be used for various purposes such as data augmentation, testing, or analysis in the TAU2 benchmark.

## Troubleshooting

### SSL Certificate Issues
If you encounter SSL certificate verification errors when downloading models:

1. **Temporary Solution (Development Only)**:
   ```python
   import ssl
   ssl._create_default_https_context = ssl._create_unverified_context
   ```

2. **Alternative Models**:
   - The notebook automatically falls back to TF-IDF if SentenceTransformer fails
   - TF-IDF provides good results for text similarity without requiring internet access

3. **Manual Model Download**:
   ```bash
   # Download model manually if needed
   pip install sentence-transformers --trusted-host pypi.org --trusted-host pypi.python.org --trusted-host files.pythonhosted.org
   ```

4. **Corporate Network**:
   If behind a corporate firewall, contact your IT department for SSL certificate configuration.

### Performance Comparison
- **SentenceTransformer**: Better semantic understanding, requires internet for first-time download
- **TF-IDF**: Fast, works offline, good for keyword-based similarity

The notebook automatically detects which method was used and adapts the similarity computation accordingly.